# Наукова міграція квантових фізиків (1900–1965)
**Матеріали до дослідження Шевчук–Гуртова.**  
**Реалізація кількісного аналізу: Оксана Гуртова.**

У цьому ноутбуці наведено код для **7 рисунків статті**:

- **Рисунки 1–2 (теплові карти)** — **оригінальний стиль Plotly / OpenStreetMap**. Після запуску комірки відкриваються в браузері за локальною адресою `http://127.0.0.1:...`. HTML-файли не створюються: сторінки передаються браузеру з пам'яті через локальний сервер, який працює у фоновому потоці ядра Jupyter.
- **Рисунки 3–4 (міграція й наставництво)** — відображаються **під комірками** ноутбука.
- **Рисунки 5–7 (APS/DPG, ВВП, TLCC)** — відображаються **під комірками** лише після підключення приватних архівних CSV.

Код **не зберігає** графіків, HTML, таблиць або інших результатів виконання у файли. Папка `figures/` містить окремо підготовлені ілюстрації; перші дві — **оригінальні невеликі зображення для попереднього перегляду** з Word, тоді як інтерактивна карта відкривається через localhost.

> **Публікація:** перш ніж додавати `.ipynb` на GitHub, очистіть результати комірок: збережені в ноутбуці зображення/таблиці можуть розкривати архівні дані. Архівні CSV не включені в репозиторій.


## 0. Підготовка середовища

Запустіть комірку нижче. Ноутбук працює, якщо його відкрили з кореня репозиторію або з папки `notebooks/`. Встановлення бібліотек описано в кореневому `README.md`.

In [ ]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
PRIVATE = ROOT / "private_data"  # Ця папка не додається до GitHub
assert (DATA / "scientists.csv").is_file(), "Не знайдено data/scientists.csv: відкрийте ноутбук із репозиторію"
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 115
print("Дані з:", DATA)
print("Усі рисунки будуть показані під комірками; автоматичного запису файлів немає.")

## 1. Авторські бази даних

`scientists.csv` — 43 біографічні записи; `locations.csv` — 481 запис про навчання, роботу й перебування (включно з історично уніфікованими країнами); `publications.csv` — 570 бібліографічних записів. Остання база збережена для прозорості, але **не використовується для побудови графіка «до/після міграції»**: такий графік вилучено зі статті через неповноту даних.

In [ ]:
scientists = pd.read_csv(DATA / 'scientists.csv', encoding='utf-8-sig')
locations = pd.read_csv(DATA / 'locations.csv', encoding='utf-8-sig')
publications = pd.read_csv(DATA / 'publications.csv', encoding='utf-8-sig')
assert len(scientists) == 43 and scientists['id_вченого'].nunique() == 43
assert len(locations) == 481
assert len(publications) == 570
assert 'Країна_уніфікована' in locations.columns
assert set(locations['id_вченого'].dropna()).issubset(set(scientists['id_вченого']))
assert set(publications['id_вченого'].dropna()).issubset(set(scientists['id_вченого']))
print('Вчені:', len(scientists), '| Локації:',len(locations),'| Публікації:',len(publications))
print('Записи із відсутніми координатами:', int(locations[['Широта','Довгота']].isna().any(axis=1).sum()))

display(scientists.head(3))

## 2. Географія квантової фізики — інтерактивні теплові карти

**Рисунки 1 і 2 статті.** Повернуто **твою початкову реалізацію** з `Period_1.py` і `Period_2.py`: `plotly.express.density_map`, підкладка `open-street-map`, кольорова шкала **Inferno**, `radius=25`, початкові `center` та `zoom`, сині маркери з підказками. Дані читаються з `data/locations.csv`.

Карти **відкриваються в окремій вкладці браузера** на локальному сервері Jupyter/Python. Це **не інтернет-хостинг**: адреса доступна тільки на цьому комп'ютері, поки запущене ядро ноутбука. HTML-файли не створюються; для завантаження підкладки OpenStreetMap потрібне підключення до інтернету. Якщо вкладка не відкрилася автоматично, натисніть надруковане посилання.


In [ ]:
# Потрібні лише pandas і Plotly; решта імпортів — стандартна бібліотека Python.
import plotly.express as px
import plotly.graph_objects as go
import webbrowser
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from IPython.display import HTML

# Один сервер для обох карт. HTML зберігаємо лише в оперативній пам'яті.
_MAP_PAGES = globals().get('_MAP_PAGES', {})
_MAP_SERVER = globals().get('_MAP_SERVER', None)


class _MapHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        path = self.path.split('?', 1)[0]
        html = _MAP_PAGES.get(path)
        if html is None:
            self.send_error(404, 'Not found')
            return
        self.send_response(200)
        self.send_header('Content-Type', 'text/html; charset=utf-8')
        self.send_header('Cache-Control', 'no-store')
        self.send_header('Content-Length', str(len(html)))
        self.end_headers()
        self.wfile.write(html)

    def log_message(self, *args):
        pass  # не заповнюємо вихід Jupyter технічними повідомленнями


def open_original_map(start_year, end_year):
    # Незмінні правила підрахунку з оригінальних Period_1.py і Period_2.py.
    full_stats = (locations.dropna(subset=['Широта', 'Довгота'])
                  .groupby('Місто(опціонально)')['id_вченого'].nunique())
    global_max = full_stats.max()

    mask = ((locations['Рік_початку'] <= end_year)
            & (locations['Рік_кінця'] >= start_year))
    filtered = locations[mask].dropna(subset=['Широта', 'Довгота'])
    city_stats = filtered.groupby('Місто(опціонально)').agg({
        'Широта': 'first',
        'Довгота': 'first',
        'id_вченого': 'nunique',
        'Країна': 'first'
    }).reset_index()
    city_stats.columns = ['Місто', 'Широта', 'Довгота', 'Кількість_вчених', 'Країна']

    fig = px.density_map(
        city_stats,
        lat='Широта', lon='Довгота', z='Кількість_вчених',
        radius=25,
        center=dict(lat=50, lon=-42),
        zoom=2.2,
        map_style='open-street-map',
        opacity=0.75,
        hover_name='Місто',
        hover_data={'Країна': True, 'Кількість_вчених': True},
        color_continuous_scale='Inferno',
        range_color=[0, global_max],
        labels={'Кількість_вчених': 'Кількість вчених'},
        title=f'Географія квантової фізики: концентрація вчених ({start_year}–{end_year})'
    )
    fig.add_trace(go.Scattermap(
        lat=city_stats['Широта'],
        lon=city_stats['Довгота'],
        mode='markers',
        marker=dict(size=6, color='blue', opacity=0.4, symbol='circle'),
        text=(city_stats['Місто'] + ', ' + city_stats['Країна'] + ': '
              + city_stats['Кількість_вчених'].astype(str) + ' вч.'),
        hoverinfo='text'
    ))

    # Замість fig.write_html(...) — тільки HTML у пам'яті.
    path = f'/map_{start_year}_{end_year}'
    _MAP_PAGES[path] = fig.to_html(full_html=True, include_plotlyjs=True)
    _MAP_PAGES[path] = _MAP_PAGES[path].encode('utf-8')

    global _MAP_SERVER
    if _MAP_SERVER is None:
        _MAP_SERVER = ThreadingHTTPServer(('127.0.0.1', 0), _MapHandler)
        threading.Thread(target=_MAP_SERVER.serve_forever, daemon=True).start()

    url = f'http://127.0.0.1:{_MAP_SERVER.server_port}{path}'
    print(f'Інтерактивна карта {start_year}–{end_year}: {url}')
    print('Вона доступна, поки працює ядро Jupyter. Файли не створюються.')
    display(HTML(f'<a href="{url}" target="_blank">Відкрити карту {start_year}–{end_year} у браузері</a>'))
    webbrowser.open_new_tab(url)


In [ ]:
# Оригінальна теплова карта 1900–1932 років (окрема вкладка браузера)
open_original_map(1900, 1932)

In [ ]:
# Оригінальна теплова карта 1945–1965 років (окрема вкладка браузера)
open_original_map(1945, 1965)

## 3. Сальдо подій академічних переміщень

**Рисунок 3 статті.** Переїзди основної академічної локації, а **не кількість різних людей**. Короткі візити та зміни історичних назв країни без переїзду не враховуються. Результати Німеччини й США — **окремі сальдо**, не доведені маршрути Німеччина → США.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================
# 1. ЗАВАНТАЖЕННЯ ТА ПІДГОТОВКА ДАНИХ
# ==========================================
# Фінальна база з історично уніфікованою назвою країни (9 колонок).
DATA_FILE = DATA / "locations.csv"
df_loc = pd.read_csv(DATA_FILE)
df_loc = df_loc.sort_values(['id_вченого', 'Рік_початку'], kind='stable')

# ==========================================
# 2. ВИЗНАЧЕННЯ МІЖДЕРЖАВНИХ ПЕРЕМІЩЕНЬ
# ==========================================
def тип_локації(опис):
    """Визначає роль біографічного запису без додавання колонок до CSV."""
    опис = str(опис).lower()

    # Народження й дитинство задають початкову країну, але не академічний переїзд.
    if 'народження' in опис or 'дитинство' in опис or 'навчання (школа)' in опис or опис == 'школа':
        return 'початкова_локація'

    # Короткі візити, стажування і другорядні події — не зміна основної локації.
    if any(слово in опис for слово in (
        'візит', 'стажуван', 'турне', 'подорож', 'лекції',
        'відпочинок', 'поховання', 'інтернування', 'випробування',
        'наукове відкриття', 'захист дисертації', 'захист кандидатської',
        'військова служба', 'заслання', 'включений до групи',
        'запрошений професор', 'запрошений дослідник',
    )):
        return 'пропустити'
    if 'еміграц' in опис or 'втеча' in опис:
        return 'вимушене_переміщення'
    if any(слово in опис for слово in (
        'робота', 'навчання', 'проживання', 'повернення', 'заснування інституту',
    )):
        return 'основна_локація'
    return 'пропустити'


def та_сама_держава(країна_1, країна_2):
    if країна_1 == країна_2:
        return True
    # Не трактуємо зміну державності на території колишньої імперії
    # як міжнародний переїзд без додаткового підтвердження.
    послідовні_держави = {'Російська імперія', 'УСРР', 'СРСР'}
    return країна_1 in послідовні_держави and країна_2 in послідовні_держави


movements = []
for uid, group in df_loc.groupby('id_вченого', sort=False):
    попередня = None

    for _, запис in group.iterrows():
        рік = запис['Рік_початку']
        if pd.isna(рік):
            continue

        тип = тип_локації(запис['Тип(навчання/робота)'])
        if тип == 'пропустити':
            continue

        країна = запис['Країна_уніфікована']
        if pd.isna(країна):
            # Невідомий етап саме еміграції: не вигадуємо країну відправлення
            # для наступного запису (напр. вчений №16, 1933–1934).
            if тип == 'вимушене_переміщення':
                попередня = None
            continue

        країна = країна.strip()
        рік = int(рік)
        кінець = запис['Рік_кінця']

        if попередня is not None:
            стара_країна = попередня['країна']
            стара_локація = попередня['рядок']
            стара_широта, стара_довгота = стара_локація['Широта'], стара_локація['Довгота']
            та_сама_локація = (
                pd.notna(стара_широта) and pd.notna(стара_довгота)
                and pd.notna(запис['Широта']) and pd.notna(запис['Довгота'])
                and abs(стара_широта - запис['Широта']) < 0.01
                and abs(стара_довгота - запис['Довгота']) < 0.01
            )

            if та_сама_локація or та_сама_держава(стара_країна, країна):
                # Перейменування країни чи пересування всередині держави.
                попередня = {'країна': країна, 'кінець': кінець, 'тип': тип, 'рядок': запис}
                continue

            # Якщо попереднє основне місце роботи досі діє, новий запис
            # може бути паралельною посадою, а не зміною основної локації.
            if (
                тип != 'вимушене_переміщення'
                and попередня['тип'] == 'основна_локація'
                and pd.notna(попередня['кінець'])
                and рік < попередня['кінець']
            ):
                continue

            # Подія у фокусному періоді. Попередні роки використовуємо
            # як контекст, але не рахуємо їх у сальдо.
            if тип != 'початкова_локація' and 1900 <= рік <= 1965:
                movements.append({
                    'Рік': рік,
                    'З_країни': стара_країна,
                    'В_країну': країна,
                })

        попередня = {'країна': країна, 'кінець': кінець, 'тип': тип, 'рядок': запис}

# У разі відсутності подій стовпці все одно існують.
df_mov = pd.DataFrame(movements, columns=['Рік', 'З_країни', 'В_країну'])
df_mov['Десятиліття'] = (df_mov['Рік'] // 10) * 10

# ==========================================
# 3. ОБЧИСЛЕННЯ САЛЬДО (ПРИТОК мінус ВІДТІК)
# ==========================================
inflow = df_mov.groupby(['Десятиліття', 'В_країну']).size().reset_index(name='Приїхало')
inflow = inflow.rename(columns={'В_країну': 'Країна'})

outflow = df_mov.groupby(['Десятиліття', 'З_країни']).size().reset_index(name='Поїхало')
outflow = outflow.rename(columns={'З_країни': 'Країна'})

net_mig = pd.merge(inflow, outflow, on=['Десятиліття', 'Країна'], how='outer').fillna(0)
net_mig['Сальдо'] = net_mig['Приїхало'] - net_mig['Поїхало']

plot_data = net_mig[net_mig['Країна'].isin(['Німеччина', 'США'])].copy()
plot_data['Десятиліття'] = plot_data['Десятиліття'].astype(int)

pivot_df = plot_data.pivot(index='Десятиліття', columns='Країна', values='Сальдо').fillna(0)
all_decades = np.arange(1900, 1970, 10)
pivot_df = pivot_df.reindex(index=all_decades, columns=['Німеччина', 'США']).fillna(0)

# ==========================================
# 4. ВІЗУАЛІЗАЦІЯ
# ==========================================
fig, ax = plt.subplots(figsize=(10, 6))
width = 0.35
x = np.arange(len(pivot_df.index))

rects1 = ax.bar(x - width / 2, pivot_df['Німеччина'], width, label='Німеччина', color='#e74c3c')
rects2 = ax.bar(x + width / 2, pivot_df['США'], width, label='США', color='#2980b9')

ax.set_ylabel('Сальдо переміщень (прибуття мінус вибуття)', fontsize=12)
ax.set_title('Сальдо змін основної академічної локації: Німеччина та США (1900–1965)', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([f'{year}-ті' if year < 1960 else '1960–1965' for year in pivot_df.index])
ax.legend(fontsize=11)
ax.axhline(0, color='black', linewidth=1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
display(fig)
plt.close(fig)

# ==========================================
# 5. ГЕНЕРАЦІЯ ТЕКСТУ ДЛЯ СТАТТІ
# ==========================================
net_ger_30s = int(pivot_df.loc[1930, 'Німеччина'])
net_usa_30s = int(pivot_df.loc[1930, 'США'])
net_ger_20s = int(pivot_df.loc[1920, 'Німеччина'])

print('\n' + '=' * 75)
print('Сальдо зафіксованих змін основної академічної локації у вибірці з 43 вчених.')
print('Розрахунок охоплює 1900–1965 роки; візити та стажування не включено.')
print('Кожний перехід враховано як подію, а не як унікальну особу.')
print(f'Німеччина, 1920-ті: {net_ger_20s:+d}.')
print(f'Німеччина, 1930-ті: {net_ger_30s:+d}.')
print(f'США, 1930-ті: {net_usa_30s:+d}.')
print('Ці значення описують лише вибірку і не доводять причин міграції '
      'або її впливу на економічний розвиток.')
print('=' * 75 + '\n')

## 4. Мережа академічного наставництва

**Рисунок 4 статті.** Орієнтовані ребра «науковий керівник → учень». Хаби визначено як вузли з **щонайменше трьома різними учнями** (`out_degree ≥ 3`). Нижче показано мережу; її розташування може відрізнятися від редакційної картинки, але зв'язки та вузли беруться з тієї ж авторської бази.

In [ ]:
# Рисунок 4. Мережа академічного наставництва — вихідний код автора.
# Використовується data/scientists.csv, графік показується під коміркою.
from pathlib import Path
import re
import textwrap
import unicodedata

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
import networkx as nx
import pandas as pd
from IPython.display import display

NAME_ALIASES = {
    "Арнольд Зоммерфельд": "Арнольд Йоганнес Вільгельм Зоммерфельд",
    "Вернер Гейзенберг": "Вернер Карл Гейзенберг",
    "Вернер Гайзенберг": "Вернер Карл Гейзенберг",
    "Юджин Вігнер": "Юджин Пол Вігнер",
    "Нільс Бор": "Нільс Генрик Давид Бор",
    "Джуліан Швінгер": "Джуліан Сеймур Швінгер",
    "Ганс Бете": "Ганс Альбрехт Бете",
    "Артур Комптон": "Артур Голлі Комптон",
}

SHORT_LABELS = {
    1: "Макс Планк", 2: "Арнольд Зоммерфельд", 3: "Альберт Ейнштейн",
    4: "Макс Борн", 5: "Нільс Бор", 6: "Ервін Шредінгер",
    7: "Луї де Бройль", 8: "Вольфганг Паулі", 9: "Вернер Гейзенберг",
    10: "Поль Дірак", 11: "Паскуаль Йордан", 12: "Юджин Вігнер",
    13: "Джон фон Нейман", 14: "Енріко Фермі", 15: "Ганс Бете",
    16: "Фелікс Блох", 17: "Еміліо Сегре", 18: "Віктор Вайскопф",
    19: "Рудольф Паєрлс", 20: "Лео Сілард", 21: "Едвард Теллер",
    22: "Гендрік Крамерс", 23: "Роберт Оппенгеймер", 24: "Річард Фейнман",
    25: "Джуліан Швінгер", 26: "Шинічіро Томонага", 27: "Маррі Гелл-Манн",
    28: "Луїс Альварес", 29: "Едвард Кондон", 30: "Ісидор Рабі",
    31: "Артур Комптон", 32: "Фрімен Дайсон", 33: "Шелдон Ґлешоу",
    34: "Стівен Вайнберг", 35: "Абдус Салам", 36: "Джон Бардін",
    37: "Лев Ландау", 38: "Ігор Тамм", 39: "Андрій Сахаров",
    40: "Яків Зельдович", 41: "Оге Бор", 42: "Чжундао Лі",
    43: "Чень-Нін Ян",
}

# Стримане чорно-біле оформлення для наукової публікації.
BG = "#FFFFFF"
INK = "#202020"
MUTED = "#686868"
LINE = "#737373"
NODE_BORDER = "#777777"
FAINT = "#F3F3F3"


def clean_name(value):
    value = str(value)
    for symbol in ("\u200b", "\u200c", "\u200d", "\u2060", "\ufeff"):
        value = value.replace(symbol, "")
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", value)).strip()


def build_graph(data):
    needed = {"id_вченого", "ПІБ", "Науковий_керівник"}
    if needed - set(data.columns):
        raise ValueError("У CSV бракує колонок: " + ", ".join(sorted(needed - set(data.columns))))
    if data["id_вченого"].isna().any() or data["id_вченого"].duplicated().any():
        raise ValueError("id_вченого має бути унікальним і непорожнім")

    graph = nx.DiGraph()
    scientist_lookup = {}
    for _, row in data.iterrows():
        sid = int(row["id_вченого"])
        full_name = clean_name(row["ПІБ"])
        if not full_name or full_name.casefold() == "nan":
            raise ValueError(f"Порожнє ПІБ: ID {sid}")
        if full_name.casefold() in scientist_lookup:
            raise ValueError(f"Повторене ПІБ: {full_name}")
        key = f"scientist:{sid}"
        scientist_lookup[full_name.casefold()] = key
        graph.add_node(key, name=full_name,
                       label=SHORT_LABELS.get(sid, full_name),
                       in_sample=True)

    aliases = {clean_name(k).casefold(): clean_name(v) for k, v in NAME_ALIASES.items()}
    for variant, canonical in aliases.items():
        if canonical.casefold() not in scientist_lookup:
            raise ValueError(f"Не знайдено вченого для псевдоніма «{variant}»")

    for _, row in data.iterrows():
        student = f"scientist:{int(row['id_вченого'])}"
        advisors = row["Науковий_керівник"]
        if pd.isna(advisors) or not str(advisors).strip():
            continue
        for name in str(advisors).split(","):
            source_name = clean_name(name)
            if not source_name:
                continue
            normalized_name = aliases.get(source_name.casefold(), source_name)
            supervisor = scientist_lookup.get(normalized_name.casefold())
            if supervisor is None:
                supervisor = f"external:{normalized_name.casefold()}"
                graph.add_node(supervisor, name=normalized_name,
                               label=normalized_name, in_sample=False)
            if supervisor == student or graph.has_edge(supervisor, student):
                raise ValueError(f"Некоректний або повторений зв'язок: {normalized_name} → {graph.nodes[student]['name']}")
            graph.add_edge(supervisor, student)
    if not nx.is_directed_acyclic_graph(graph):
        raise ValueError("Помилка: виявлено цикл у наставництві")
    return graph


def layers_of(graph, component):
    sub = graph.subgraph(component)
    levels = {}
    for node in nx.topological_sort(sub):
        parents = list(sub.predecessors(node))
        levels[node] = 0 if not parents else max(levels[p] for p in parents) + 1
    return [[n for n in sorted(sub.nodes, key=lambda x: graph.nodes[x]["label"])
             if levels[n] == layer] for layer in range(max(levels.values()) + 1)]


def wrap_name(label, width=18):
    """Без перенесення слів посередині; довгі імена у дві строки."""
    return textwrap.fill(label, width=width, break_long_words=False,
                         break_on_hyphens=False)


def node_card(ax, x, y, w, h, node, graph, hubs, font_size=9.1):
    """Простий прямокутний вузол; хаби виділені лише товщиною рамки."""
    is_hub = node in hubs
    in_sample = graph.nodes[node]["in_sample"]
    fill = BG if in_sample else FAINT
    edge = INK if is_hub else NODE_BORDER
    rect = Rectangle(
        (x - w / 2, y - h / 2), w, h,
        transform=ax.transData,
        facecolor=fill,
        edgecolor=edge,
        linewidth=1.65 if is_hub else 0.85,
        zorder=4,
    )
    ax.add_patch(rect)
    label = wrap_name(graph.nodes[node]["label"], 12 if w < .29 else 17)
    ax.text(
        x, y, label, ha="center", va="center", fontsize=font_size,
        color=INK if in_sample else "#444444",
        fontweight="bold" if is_hub else "normal",
        linespacing=1.1, zorder=5,
    )
    return rect


def component_panel(ax, graph, component, hubs, heading=None, compact=False):
    """Незалежна схема без декоративної картки й зайвого підзаголовка."""
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    layers = layers_of(graph, component)
    widest = max(map(len, layers))
    card_w = .205 if widest >= 4 else (.243 if widest == 3 else .29)
    card_h = .21 if compact else (.128 if len(layers) >= 4 else .152)
    font_size = 9.5 if not compact else 9.3
    if compact and widest >= 3:
        font_size = 8.8

    y_top, y_bottom = .85, .15
    positions = {}
    for level, nodes in enumerate(layers):
        y = (y_top + y_bottom) / 2 if len(layers) == 1 else (
            y_top - level * (y_top - y_bottom) / (len(layers) - 1)
        )
        count = len(nodes)
        if count == 1:
            xs = [.5]
        else:
            margin = card_w / 2 + .052
            xs = [margin + j * ((1 - 2 * margin) / (count - 1))
                  for j in range(count)]
        for node, x in zip(nodes, xs):
            positions[node] = (x, y)

    patches = {
        n: node_card(ax, *positions[n], card_w, card_h, n, graph, hubs, font_size)
        for n in component
    }
    for source, target in graph.subgraph(component).edges:
        arrow = FancyArrowPatch(
            positions[source], positions[target],
            patchA=patches[source], patchB=patches[target],
            arrowstyle="-|>", mutation_scale=10.5,
            linewidth=.95, color=LINE,
            connectionstyle="arc3,rad=0", zorder=2,
        )
        ax.add_patch(arrow)


def add_heading(fig, title):
    fig.text(.055, .96, title, fontsize=17.5, color=INK,
             family="DejaVu Serif", weight="normal", va="top")


def make_main_figure(graph, components, hubs):
    """Основні компоненти мережі: схема без декоративного оформлення."""
    fig = plt.figure(figsize=(16, 11.4), facecolor=BG)
    add_heading(fig, "Мережа академічного наставництва")
    grid = fig.add_gridspec(
        nrows=3, ncols=6,
        left=.055, right=.945, top=.87, bottom=.075,
        hspace=.27, wspace=.24, height_ratios=[1.15, 1.15, .9]
    )
    for i, component in enumerate(components[:4]):
        ax = fig.add_subplot(grid[i // 2, (i % 2) * 3 : (i % 2) * 3 + 3])
        component_panel(ax, graph, component, hubs)
    for i, component in enumerate(components[4:7]):
        ax = fig.add_subplot(grid[2, i * 2 : (i + 1) * 2])
        component_panel(ax, graph, component, hubs, compact=True)
    return fig


# Тільки рисунок зі статті; без другої сторінки та текстових таблиць.
data = pd.read_csv(DATA / "scientists.csv", encoding="utf-8-sig")
graph = build_graph(data)
hubs = {n for n in graph.nodes if graph.out_degree(n) >= 3}
components = sorted(nx.weakly_connected_components(graph),
                    key=lambda c: (-len(c), min(c)))
fig = make_main_figure(graph, components, hubs)
display(fig)
plt.close(fig)


## 5. Архівні ряди фізичних товариств і ВВП — локальний запуск

**Рисунки 5 і 6 статті:** США (APS/ВВП) та Німеччина (DPG/регіональні товариства/ВВП). **В публічному ZIP цих CSV немає.**

Щоб побачити два графіки прямо під комірками, помістіть **дозволені для локального використання** копії `usa_data.csv` і `germany_data.csv` у приватну папку `private_data/` на своєму комп'ютері та перезапустіть секцію. Ні файли даних, ні побудовані графіки не зберігаються автоматично. **Не публікуйте ноутбук зі збереженими приватними результатами.**

In [ ]:
GERMANY_FILE = PRIVATE / 'germany_data.csv'
USA_FILE = PRIVATE / 'usa_data.csv'
HAVE_PRIVATE = GERMANY_FILE.is_file() and USA_FILE.is_file()
print('Архівні файли:', 'знайдено — графіки й аналіз будуть показані локально' if HAVE_PRIVATE
      else 'не надано — приватні розділи будуть пропущені')

### APS/ВВП, США

In [ ]:
if HAVE_PRIVATE:
    import pandas as pd
    import matplotlib.pyplot as plt
    import numpy as np

    # --- 1. Завантаження даних ---
    file_path = USA_FILE
    df = pd.read_csv(file_path)

    # --- 2. Автовизначення колонок ---
    year_col = None
    phys_col = None
    gdp_col = None

    for col in df.columns:
        col_lower = col.lower()
        if 'рік' in col_lower or 'year' in col_lower or 'pik' in col_lower:
            year_col = col
        elif 'aps' in col_lower or 'фізик' in col_lower or 'physicist' in col_lower:
            phys_col = col
        elif 'gdp' in col_lower or 'ввп' in col_lower:
            gdp_col = col

    if year_col is None: year_col = df.columns[0]
    if phys_col is None: phys_col = df.columns[1]
    if gdp_col is None: gdp_col = df.columns[2]

    # --- 3. Приведення даних до числового формату ---
    df[gdp_col] = df[gdp_col].astype(str).str.replace(',', '.').str.replace(' ', '').str.replace('$', '')
    df[gdp_col] = pd.to_numeric(df[gdp_col], errors='coerce')
    df[phys_col] = pd.to_numeric(df[phys_col], errors='coerce')

    df_clean = df.dropna(subset=[year_col, phys_col, gdp_col])
    df_clean = df_clean.sort_values(year_col)

    # --- 4. Графік з двома осями Y ---
    fig, ax1 = plt.subplots(figsize=(12, 7))

    color1 = '#1f77b4'
    ax1.set_xlabel('Рік', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Кількість фізиків (члени APS)', fontsize=12, fontweight='bold', color=color1)
    line1 = ax1.plot(df_clean[year_col], df_clean[phys_col], 
                     marker='o', linestyle='-', linewidth=2, markersize=6,
                     color=color1, label='Кількість фізиків (APS)')
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.grid(True, linestyle='--', alpha=0.3)

    ax2 = ax1.twinx()
    color2 = '#d62728'
    ax2.set_ylabel('ВВП на душу населення (2011 $)', fontsize=12, fontweight='bold', color=color2)
    line2 = ax2.plot(df_clean[year_col], df_clean[gdp_col], 
                     marker='s', linestyle='-', linewidth=2, markersize=6,
                     color=color2, label='ВВП на душу населення')
    ax2.tick_params(axis='y', labelcolor=color2)

    # --- 5. Налаштування шкали ВВП ---
    gdp_min = df_clean[gdp_col].min()
    gdp_max = df_clean[gdp_col].max()
    step = 2000
    gdp_ticks = np.arange(np.floor(gdp_min / step) * step, np.ceil(gdp_max / step) * step + step, step)
    ax2.set_yticks(gdp_ticks)
    ax2.set_ylim(gdp_min - 500, gdp_max + 500)

    # --- 6. Назва, легенда, оформлення ---
    plt.title('Кількість фізиків та ВВП на душу населення: США (1900–1965)', fontsize=14, fontweight='bold')
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left', fontsize=10)
    ax1.set_xlim(df_clean[year_col].min() - 2, df_clean[year_col].max() + 2)
    ax2.grid(False)
    plt.tight_layout()

    # --- 7. Збереження ---
    display(fig)


    plt.close(fig)

    print("APS/ВВП, США: зображено вище, файли не створювалися")
else:
    print("Графік USA: пропущено (немає архівних CSV)")

### DPG/ВВП, Німеччина

In [ ]:
if HAVE_PRIVATE:
    import pandas as pd
    import matplotlib.pyplot as plt
    import numpy as np

    # --- 1. Завантаження даних ---
    file_path = GERMANY_FILE
    df = pd.read_csv(file_path)

    # --- 2. Очищення даних (перетворення на числа) ---
    def clean_numeric(value):
        if isinstance(value, str):
            value = value.replace(',', '').replace(' ', '').strip()
        try:
            return float(value)
        except (ValueError, TypeError):
            return np.nan

    for col in df.columns:
        if col != df.columns[0]: 
            df[col] = df[col].apply(clean_numeric)

    # --- 3. Автовизначення колонок ---
    year_col = df.columns[0]
    phys_col = None
    gdp_col = None

    for col in df.columns:
        col_lower = col.lower()
        if 'dpg' in col_lower or 'фізик' in col_lower or 'physicist' in col_lower:
            phys_col = col
        elif 'gdp' in col_lower or 'ввп' in col_lower:
            gdp_col = col

    if phys_col is None: phys_col = df.columns[1]
    if gdp_col is None: gdp_col = df.columns[2]

    # --- 4. Підготовка даних ---
    df_clean = df.dropna(subset=[year_col, phys_col, gdp_col]).copy()
    df_clean[year_col] = df_clean[year_col].astype(int)
    df_clean = df_clean.sort_values(year_col)

    # --- 5. Графік з двома осями Y ---
    fig, ax1 = plt.subplots(figsize=(12, 7))

    color1 = '#1f77b4'
    ax1.set_xlabel('Рік', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Кількість фізиків (члени DPG)', fontsize=12, fontweight='bold', color=color1)
    line1 = ax1.plot(df_clean[year_col], df_clean[phys_col], 
                     marker='o', linestyle='-', linewidth=2, markersize=6,
                     color=color1, label='Кількість фізиків (DPG)')
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.grid(True, linestyle='--', alpha=0.3)

    ax2 = ax1.twinx()
    color2 = '#d62728'
    ax2.set_ylabel('ВВП на душу населення (2011 $)', fontsize=12, fontweight='bold', color=color2)
    line2 = ax2.plot(df_clean[year_col], df_clean[gdp_col], 
                     marker='s', linestyle='-', linewidth=2, markersize=6,
                     color=color2, label='ВВП на душу населення')
    ax2.tick_params(axis='y', labelcolor=color2)

    # --- 6. Налаштування шкали ВВП ---
    gdp_min = df_clean[gdp_col].min()
    gdp_max = df_clean[gdp_col].max()
    step = 2000
    gdp_round_min = np.floor(gdp_min / step) * step
    gdp_round_max = np.ceil(gdp_max / step) * step
    ax2.set_ylim(gdp_round_min, gdp_round_max)
    ax2.set_yticks(np.arange(gdp_round_min, gdp_round_max + step, step))

    # --- 7. Назва та легенда ---
    plt.title('Кількість фізиків та ВВП на душу населення: Німеччина (1900–1965)', fontsize=14, fontweight='bold')
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left', fontsize=10)
    ax1.set_xlim(df_clean[year_col].min() - 2, df_clean[year_col].max() + 2)
    ax2.grid(False)
    plt.tight_layout()

    # --- 8. Збереження ---
    display(fig)


    plt.close(fig)

    print("DPG/ВВП, Німеччина: зображено вище, файли не створювалися")
else:
    print("Графік Germany: пропущено (немає архівних CSV)")

## 6. TLCC, перевірка стаціонарності та прогностичних зв'язків — локально

**Рисунок 7 статті** — крос-кореляція зі зсувом **−10…+10 років**. У цьому ж розділі обчислюються ADF, Тода—Ямамото і поправка Холма. Найвищий лаг кореляції **не є** доведеною затримкою історичного впливу. Публічний ноутбук містить код, але **не** архівні ряди і **не** готові приватні графіки чи таблиці.

Для перевірки чутливості наведено порівняння повного лінійно інтерпольованого ряду, наявних спостережень та PCHIP. **Інтерполяція не відновлює пропущених фактів.**

In [ ]:
"""TLCC + перевірка прогностичних зв'язків Грейнджера, 1900–1965 рр.

ВХІД (у тій самій папці): germany_data.csv, usa_data.csv.
Допускаються також назви germany_data(1).csv / usa_data(1).csv.
Первинні файли не редагуються.

Дослідження:
  1) TLCC на початкових рівнях та контроль БЕЗ інтерполяції;
  2) ADF на логарифмах рядів і їхніх різницях;
  3) розширений VAR / тест Тоди—Ямамото (перші k лагів, k+dmax допоміжних);
  4) класичний тест Грейнджера на стаціонаризованих логарифмах (контрольний);
  5) Holm-поправка для чотирьох основних гіпотез та чутливість до вибору k.

ВАЖЛИВО: часовий/прогностичний зв'язок НЕ доводить економічну причинність.
Зв'язок перевіряється між ВВП НА ДУШУ та членством у товариствах,
а НЕ загальною кількістю фізиків.

За замовчуванням результати зберігаються тільки в новій папці
TLCC_results поруч зі скриптом; початкові CSV залишаються без змін.
"""

from pathlib import Path
from contextlib import redirect_stdout
from io import StringIO
import inspect
import unicodedata
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

START_YEAR, END_YEAR = 1900, 1965
MAX_TLCC_LAG = 10
MIN_PAIRS = 10
MAX_VAR_K = 5
MAX_GRANGER_LAG = 5
ALPHA = 0.05
WHITENESS_LAG = 12

BASE_DIR = PRIVATE
GDP_COLUMN = "GDP_per_capita(in 2011$)"
SOURCES = {
    "Німеччина": {"name": "germany_data.csv", "member": "DPG", "color": "#315469"},
    "США": {"name": "usa_data.csv", "member": "APS", "color": "#9C5D51"},
}
SEPARATOR = "═" * 72


def find_source(name):
    """Дозволяє (1) у назвах і Unicode NFC/NFD на macOS."""
    stem = Path(name).stem
    allowed = {unicodedata.normalize("NFC", x)
               for x in (name, stem + "(1).csv")}
    matches = [p for p in BASE_DIR.glob("*.csv")
               if unicodedata.normalize("NFC", p.name) in allowed]
    if not matches:
        raise FileNotFoundError(f"Не знайдено {name} у {BASE_DIR}")
    matches.sort(key=lambda p: 0 if unicodedata.normalize("NFC", p.name) == name else 1)
    return matches[0]


def parse_number(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (float, int, np.number)):
        return float(val)
    s = str(val).strip().replace("\u00a0", "").replace("\u202f", "").replace(" ", "")
    if not s or s in {"—", "–", "-"}:
        return np.nan
    if "," in s and "." in s:
        s = s.replace(",", "") if s.rfind(".") > s.rfind(",") else s.replace(".", "").replace(",", ".")
    elif "," in s:
        pieces = s.lstrip("+-").split(",")
        s = s.replace(",", "") if all(len(p) == 3 and p.isdigit() for p in pieces[1:]) else s.replace(",", ".")
    try:
        return float(s)
    except ValueError as exc:
        raise ValueError(f"Неможливо прочитати число «{val}»") from exc


def load_source(country, info):
    path = find_source(info["name"])
    df = pd.read_csv(path, encoding="utf-8-sig")
    year_candidates = [col for col in df.columns if col.strip().lower() in {"рік", "рiк", "year"}]
    if len(year_candidates) != 1:
        raise ValueError(f"{path.name}: потрібен стовпчик 'Рік'")
    year_col = year_candidates[0]
    need = [year_col, info["member"], GDP_COLUMN]
    absent = [c for c in need if c not in df.columns]
    if absent:
        raise ValueError(f"{path.name}: відсутні колонки {absent}")
    df = df[need].copy()
    df.columns = ["Year", "Membership", "GDP"]
    df["Year"] = pd.to_numeric(df["Year"], errors="raise").astype(int)
    if df["Year"].duplicated().any():
        raise ValueError(f"{path.name}: повторені роки")
    df = df.set_index("Year").sort_index().loc[START_YEAR:END_YEAR]
    if len(df) != END_YEAR - START_YEAR + 1 or set(df.index) != set(range(START_YEAR, END_YEAR+1)):
        raise ValueError(f"{path.name}: має бути по одному запису на кожний рік 1900–1965")
    for col in ["Membership", "GDP"]:
        df[col] = df[col].map(parse_number)
        if df[col].dropna().le(0).any():
            raise ValueError(f"{path.name}: не можна брати логарифм не-додатних значень {col}")

    raw = df.copy()
    prepared = df.interpolate(method="index", limit_area="inside")
    if prepared[["Membership", "GDP"]].isna().any().any():
        raise ValueError(f"{path.name}: залишилися пропуски після внутрішньої інтерполяції")
    counts = {col: int((raw[col].isna() & prepared[col].notna()).sum()) for col in ["Membership", "GDP"]}
    return path, raw, prepared, counts


def crosscorr(datax, datay, lag=0):
    """corr(членство[t], ВВП[t-lag]); плюс означає ВВП передує членству."""
    return datax.corr(datay.shift(lag))


def calc_tlcc(frame, country, variant):
    out = []
    for lag in range(-MAX_TLCC_LAG, MAX_TLCC_LAG+1):
        shifted_gdp = frame["GDP"].shift(lag)
        n = int(pd.concat([frame["Membership"], shifted_gdp], axis=1).dropna().shape[0])
        corr = crosscorr(frame["Membership"], frame["GDP"], lag) if n >= MIN_PAIRS else np.nan
        out.append({"Країна": country, "Дані": variant, "Лаг": lag,
                    "Кореляція_r": corr, "Пар_років": n})
    return pd.DataFrame(out)


def peak_tlcc(frame):
    f = frame.dropna(subset=["Кореляція_r"])
    return f.loc[f["Кореляція_r"].idxmax()] if len(f) else None


def direction_text(lag):
    if lag > 0:
        return f"Максимум при зсуві ВВП на +{lag} років (не причинний вплив)"
    if lag < 0:
        return f"Максимум при зсуві ВВП на {lag} років (не причинний вплив)"
    return "Найбільша кореляція за відсутності часового зсуву"


def adf_result(series):
    if series.dropna().nunique() < 3:
        return np.nan, np.nan
    args = {"regression": "c", "autolag": "AIC"}
    if "result_object" in inspect.signature(adfuller).parameters:
        args["result_object"] = False
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning,
                                message="adfuller currently returns a plain tuple.*")
        result = adfuller(series.dropna().to_numpy(), **args)
    return float(result[1]), int(result[2])


def adf_diagnostics(logframe, country, member):
    rows = []
    for col, label in [("Membership", member), ("GDP", "ВВП")]:
        for order in range(3):
            x = logframe[col]
            for _ in range(order):
                x = x.diff()
            p, used_lags = adf_result(x.dropna())
            rows.append({"Країна": country, "Ряд": label, "Порядок_різниць": order,
                         "ADF_p": p, "Лагів_ADF": used_lags,
                         "ADF_відхиляє_одиничний_корінь": bool(p < ALPHA)})
    return pd.DataFrame(rows)


def inferred_dmax(adf):
    orders = []
    for name, group in adf.groupby("Ряд", sort=False):
        candidates = group[group["ADF_p"] < ALPHA]
        if candidates.empty:
            raise ValueError(f"Для ряду {name} не отримано стаціонарності навіть у других різницях")
        orders.append(int(candidates["Порядок_різниць"].min()))
    return max(orders)


def logframe_for_var(interpolated):
    frame = np.log(interpolated[["Membership", "GDP"]].copy())
    frame.index = pd.period_range(str(START_YEAR),str(END_YEAR), freq="Y")
    return frame


def whiteness_p(frame, total_lags):
    fitted = VAR(frame).fit(total_lags, trend="c")
    return float(fitted.test_whiteness(nlags=WHITENESS_LAG).pvalue)


def choose_k(logframe, dmax):
    model = VAR(logframe)
    selected = model.select_order(MAX_VAR_K)
    bic = int(selected.selected_orders["bic"])
    k = max(bic, dmax, 1)  # правило Тоди—Ямамото потребує k >= dmax
    checks = []
    for trial in range(k, MAX_VAR_K+1):
        p = whiteness_p(logframe, trial + dmax)
        checks.append((trial, p))
        if p >= ALPHA:
            return k, trial, checks
    return k, None, checks


def ty_wald(frame, k, dmax, country, member, diagnostic=False):
    """Тода—Ямамото: VAR(k+dmax), Wald тільки на перших k лагах.

    Побудова обох рівнянь через OLS з ідентичними регресорами еквівалентна
    оцінюванню VAR(k+dmax) методом найменших квадратів. Wald ~ χ²(k).
    """
    lag_total = k+dmax
    Xcols = {}
    for lag in range(1,lag_total+1):
        for name in ("Membership", "GDP"):
            Xcols[f"{name}_L{lag}"] = frame[name].shift(lag)
    regressors = sm.add_constant(pd.DataFrame(Xcols, index=frame.index).iloc[lag_total:], has_constant="add")
    y = frame.iloc[lag_total:]
    rows=[]
    for cause,effect in [("GDP", "Membership"), ("Membership", "GDP")]:
        fitted = sm.OLS(y[effect], regressors).fit()
        R = np.zeros((k, len(regressors.columns)))
        for lag in range(1,k+1):
            R[lag-1, regressors.columns.get_loc(f"{cause}_L{lag}")] = 1
        test = fitted.wald_test(R, use_f=False, scalar=True)
        causal_name = "ВВП" if cause == "GDP" else member
        target_name = "ВВП" if effect == "GDP" else member
        rows.append({"Країна": country, "Напрям": f"{causal_name} → {target_name}",
                     "k": k, "dmax": dmax, "VAR_лагів": lag_total, "N": len(y),
                     "Wald_χ2": float(test.statistic), "df": k, "p_value": float(test.pvalue),
                     "Значущий_без_поправки": bool(test.pvalue < ALPHA)})
    return pd.DataFrame(rows)


def run_toda(logframe, country, member, dmax):
    k_bic, chosen, checks = choose_k(logframe, dmax)
    if chosen is None:
        raise ValueError(f"{country}: залишкова автокореляція VAR при всіх k={k_bic}–{MAX_VAR_K}; висновок ненадійний")
    primary = ty_wald(logframe, chosen, dmax, country, member)
    primary["BIC_k"] = k_bic
    primary["p_білий_шум"] = whiteness_p(logframe, chosen+dmax)
    primary["Вибір_k"] = "BIC" if chosen == k_bic else "BIC + усунення автокореляції залишків"

    alternatives=[]
    for k in range(max(1,dmax),MAX_VAR_K+1):
        trial=ty_wald(logframe, k, dmax, country, member)
        trial["p_білий_шум"] = whiteness_p(logframe, k+dmax)
        trial["Прийнятна_діагностика_залишків"] = trial["p_білий_шум"] >= ALPHA
        alternatives.append(trial)
    return primary, pd.concat(alternatives, ignore_index=True), checks


def standard_granger(logframe, country, member, dmax):
    """Зберігаємо оригінальний тест у коректнішій формі: на стаціонаризованих лог-рядах.
    Показуємо всі лаги 1–5 тільки як діагностичну таблицю, НЕ як п'ять
    окремих підтверджених гіпотез.
    """
    stationary = logframe.copy()
    for _ in range(dmax):
        stationary = stationary.diff().dropna()
    p1,_ = adf_result(stationary["Membership"])
    p2,_ = adf_result(stationary["GDP"])
    if not (p1 < ALPHA and p2 < ALPHA):
        raise ValueError(f"{country}: неможливо коректно застосувати стандартний Грейнджер")
    selected_k=max(1,int(VAR(stationary).select_order(MAX_GRANGER_LAG).selected_orders['bic']))
    out=[]
    for cause,effect in [("GDP", "Membership"),("Membership", "GDP")]:
        with redirect_stdout(StringIO()):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", FutureWarning)
                test=grangercausalitytests(stationary[[effect,cause]], maxlag=MAX_GRANGER_LAG)
        for k in range(1,MAX_GRANGER_LAG+1):
            f,p,df_num,df_denom = test[k][0]["ssr_ftest"]
            out.append({"Країна": country,
                        "Напрям": f"{'ВВП' if cause=='GDP' else member} → {'ВВП' if effect=='GDP' else member}",
                        "Порядок_лог_різниць": dmax, "Лаг": k, "BIC_лаг": selected_k,
                        "F": float(f), "p_value": float(p),
                        "Значущий_без_поправки": bool(p < ALPHA),
                        "BIC_основна_специфікація": k==selected_k})
    return pd.DataFrame(out)


def main_plot(tlcc_table, maxima):
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})
    fig,ax=plt.subplots(figsize=(10.8,6),facecolor='white')
    for country,info in SOURCES.items():
        series=tlcc_table[(tlcc_table["Країна"]==country)&(tlcc_table["Дані"]=="З інтерполяцією")]
        observed=tlcc_table[(tlcc_table["Країна"]==country)&(tlcc_table["Дані"]=="Лише фактичні")]
        ax.plot(series['Лаг'],series['Кореляція_r'],lw=2.2,marker='o',markersize=3.6,
                color=info['color'],label=country)
        ax.plot(observed['Лаг'],observed['Кореляція_r'],lw=1.0,ls='--',alpha=.50,
                color=info['color'])
        peak=maxima[country][0]
        ax.scatter([peak['Лаг']],[peak['Кореляція_r']],s=65,marker='o',
                   color=info['color'],edgecolors='white',linewidth=.8,zorder=5)
        ax.annotate(f"r={peak['Кореляція_r']:.3f}; k={int(peak['Лаг']):+d}",
                    xy=(peak['Лаг'],peak['Кореляція_r']),xytext=(7,-20 if country=='США' else -22),
                    textcoords='offset points', fontsize=9,color=info['color'],
                    bbox={'facecolor':'white','edgecolor':'none','alpha':0.8,'pad':1})
    ax.axvline(0,ls='--',color='0.38',lw=1)
    ax.set_xticks(np.arange(-10,11,2))
    ax.set_xlabel('Лаг k, років   (−: членство передує ВВП; +: ВВП передує членству)')
    ax.set_ylabel('Коефіцієнт Пірсона, r')
    ax.set_title('Членство у фізичних товариствах і ВВП на душу населення: TLCC, 1900–1965')
    ax.grid(axis='y',color='#E4E5E6',linewidth=.7)
    ax.legend(frameon=False,loc='lower right')
    fig.text(.5,.015,'Суцільні лінії: із лінійною інтерполяцією; пунктир: лише спостережені значення.',
             ha='center',fontsize=8.5,color='#666666')
    fig.tight_layout(rect=[0,.032,1,1])
    return fig


def print_table(df,fields,formatters=None):
    view=df[fields].copy()
    if formatters:
        for col,fmt in formatters.items():
            view[col]=view[col].map(fmt)
    print(view.to_string(index=False))


def main():
    print('\n'+SEPARATOR+'\nTLCC І ТЕСТИ ГРЕЙНДЖЕРА: РЕАЛЬНІ CSV, 1900–1965\n'+SEPARATOR)
    print('Стрілка X → Y означає: минулі X містять додаткову ПРОГНОСТИЧНУ інформацію про Y.')
    print('Точні файли зчитуються з приватної локальної папки. Немає тестових підмін.')
    tlcc_parts=[]; adf_parts=[]; primary_parts=[]; sensitivity_parts=[]; standard_parts=[]
    maximum_rows=[]; maximum_pairs={}; chosen_info=[]
    for country,info in SOURCES.items():
        print('\n'+SEPARATOR+f'\n{country.upper()}: ЧИТАННЯ ДАНИХ\n'+SEPARATOR)
        path,raw,prepared,counts = load_source(country,info)
        print('Джерело:',path.resolve())
        print('Років:',len(raw),'; інтерпольовано членство:',counts['Membership'], '; ВВП:',counts['GDP'])
        full=calc_tlcc(prepared,country,'З інтерполяцією')
        obs=calc_tlcc(raw,country,'Лише фактичні')
        tlcc_parts.extend([full,obs])
        p_full,p_obs=peak_tlcc(full),peak_tlcc(obs)
        maximum_pairs[country]=(p_full,p_obs)
        print(f'\nTLCC: r={p_full["Кореляція_r"]:.3f}; k={int(p_full["Лаг"]):+d}; пар={int(p_full["Пар_років"])}')
        print(' ',direction_text(int(p_full['Лаг'])))
        print(f'Без інтерполяції: r={p_obs["Кореляція_r"]:.3f}; k={int(p_obs["Лаг"]):+d}; пар={int(p_obs["Пар_років"])}')
        maximum_rows.append({"Країна":country,"Макс_r":p_full['Кореляція_r'],"Лаг":int(p_full['Лаг']),
                             "Пар":int(p_full['Пар_років']),"Без_інтерп_r":p_obs['Кореляція_r'],
                             "Без_інтерп_лаг":int(p_obs['Лаг']),"Без_інтерп_пар":int(p_obs['Пар_років'])})
        logged=logframe_for_var(prepared)
        diag=adf_diagnostics(logged,country,info['member'])
        adf_parts.append(diag)
        dmax=inferred_dmax(diag)
        print('\nADF(log рядів):')
        print_table(diag,['Ряд','Порядок_різниць','ADF_p'],{'ADF_p':lambda v:f'{v:.5f}'})
        print('Верхня оцінка порядку інтегрованості dmax=',dmax,
              ' (за ADF, залежить від специфікації тесту).')
        primary,other,checks=run_toda(logged,country,info['member'],dmax)
        primary_parts.append(primary)
        sensitivity_parts.append(other)
        chosen_info.append(f'{country}: BIC k={int(primary["BIC_k"].iloc[0])}, '
                           f'вибрано k={int(primary["k"].iloc[0])}, dmax={dmax}, '
                           f'портманто p={primary["p_білий_шум"].iloc[0]:.4f}.')
        print('\nТЕСТ ГРЕЙНДЖЕРА: ТОДА—ЯМАМОТО')
        print('VAR порядку k+dmax =',int(primary['VAR_лагів'].iloc[0]),
              '; k =',int(primary['k'].iloc[0]),'; dmax =',dmax)
        print_table(primary,['Напрям','Wald_χ2','df','p_value'],{'Wald_χ2':lambda v:f'{v:.3f}', 'p_value':lambda v:f'{v:.4f}'})
        standard=standard_granger(logged,country,info['member'],dmax)
        standard_parts.append(standard)
        print('\nКЛАСИЧНИЙ ГРЕЙНДЖЕР: ПОКАЗ УСІХ ЛАГІВ 1–5 (ДІАГНОСТИКА)')
        print(f'Стаціонаризовано: Δ^{dmax}log; BIC пропонує лаг {standard["BIC_лаг"].iloc[0]}.')
        for direction,part in standard.groupby('Напрям',sort=False):
            print(f'  Гіпотеза: {direction}')
            print_table(part,['Лаг','F','p_value'],{'F':lambda v:f'{v:.3f}','p_value':lambda v:f'{v:.4f}'})

    tlcc_table=pd.concat(tlcc_parts,ignore_index=True)
    tlcc_max=pd.DataFrame(maximum_rows)
    adf_table=pd.concat(adf_parts,ignore_index=True)
    primary=pd.concat(primary_parts,ignore_index=True)
    sens=pd.concat(sensitivity_parts,ignore_index=True)
    standard=pd.concat(standard_parts,ignore_index=True)
    reject,p_holm,_,_=multipletests(primary['p_value'],alpha=ALPHA,method='holm')
    primary['p_Holm']=p_holm
    primary['Після_Holm_значущий']=reject
    print('\n'+SEPARATOR+'\nПІДСУМОК: ОСНОВНИЙ ТЕСТ ГРЕЙНДЖЕРА (ТОДА—ЯМАМОТО)\n'+SEPARATOR)
    print_table(primary,['Країна','Напрям','k','dmax','Wald_χ2','p_value','p_Holm'],
                {'Wald_χ2':lambda v:f'{v:.3f}','p_value':lambda v:f'{v:.4f}', 'p_Holm':lambda v:f'{v:.4f}'})
    for _,row in primary.iterrows():
        print(f'  {row["Країна"]}, {row["Напрям"]}: '
              + ('зв’язок статистично виявлено ПІСЛЯ Holm' if row['Після_Holm_значущий']
                 else 'немає підтвердження на 5% після Holm')
              + ' (не є доказом причинності).')

    fig1=main_plot(tlcc_table,maximum_pairs)
    print('\nВАЖЛИВО: не можна ототожнювати APS/DPG із числом фізиків або p-value з доведеною причинністю.')
    display(fig1)
    plt.close(fig1)

In [ ]:
if HAVE_PRIVATE:
    main()
else:
    print("TLCC і статистичні тести не запущено: потрібні приватні CSV")

## 7. Додаткова перевірка PCHIP — лише локально

Порівняння чутливості найбільшої крос-кореляції до способу заповнення пропусків: лінійна інтерполяція, тільки наявні точки, PCHIP. Таблиця з'являється під коміркою; **файли не створюються**. Цей додаток не змінює основного TLCC і не є незалежним доказом причинності.

In [ ]:
if HAVE_PRIVATE:
    from scipy.interpolate import PchipInterpolator
    rows = []
    for country, info in SOURCES.items():
        _, raw, linear, _ = load_source(country, info)
        pchip = raw.copy()
        for col in ("Membership", "GDP"):
            vals = raw[col].dropna()
            pchip[col] = PchipInterpolator(vals.index.values.astype(float), vals.values, extrapolate=False)(raw.index.values.astype(float))
        for name, frame in (("Лінійна інтерполяція", linear),
                            ("Лише наявні значення", raw),
                            ("PCHIP", pchip)):
            best = peak_tlcc(calc_tlcc(frame, country, name))
            rows.append({"Країна": country, "Спосіб": name,
                         "Найвища r": round(float(best["Кореляція_r"]), 3),
                         "Лаг": int(best["Лаг"])})
    display(pd.DataFrame(rows))
else:
    print("PCHIP не виконано: потрібні приватні CSV")

## 8. Підсумок відтворення рисунків

- **Рисунки 1–2:** Plotly + OpenStreetMap, **відкриваються через localhost у браузері**, без запису HTML/PNG.
- **Рисунки 3–4:** Matplotlib / NetworkX, показуються під комірками.
- **Рисунки 5–7:** показуються під комірками **лише локально** з `private_data/usa_data.csv` і `private_data/germany_data.csv`.

Файл `publications.csv` — допоміжна бібліографічна база; вилучений зі статті графік публікаційної активності не генерується. Ноутбук не зберігає результати виконання у файли. При публікації **обов'язково очищайте outputs** усіх комірок, якщо ви запускали приватний аналіз.
